## 7. Explicabilidad, preguntas obligatorias y conclusión

In [ ]:
# preparación de datos (se elimina al unir con el notebook principal)
import re
import pandas as pd
import numpy as np

def leer_anio(anio):
    d = pd.read_excel(f"data/Precios{anio}.xls", sheet_name="POE (Anual)", skiprows=3, engine="xlrd")
    d.columns = ["fecha", "hora", "precio"]
    d["anio"] = anio
    return d

def a_numero(x):
    if isinstance(x, (int, float)):
        return float(x)
    s = re.sub(r"[^0-9.]", "", str(x))
    if s.count(".") > 1:
        p = s.find(".")
        s = s[:p+1] + s[p+1:].replace(".", "")
    return float(s) if s not in ("", ".") else np.nan

df = pd.concat([leer_anio(a) for a in [2023, 2024, 2025]], ignore_index=True)
df["precio"] = df["precio"].apply(a_numero)
df["hora"] = df["hora"].replace({220: 22, 230: 23, 2300: 23})
df["datetime"] = df["fecha"] + pd.to_timedelta(df["hora"], unit="h")
df = df.sort_values("datetime").reset_index(drop=True)
df["precio"] = df["precio"].interpolate()
df["mes"] = df["datetime"].dt.month
df["dia_sem"] = df["datetime"].dt.dayofweek
df["finde"] = df["dia_sem"] >= 5
train = df[df["anio"] < 2025].copy()

meses_nom = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

### 7.1 ¿Cuál es el comportamiento general del precio de oportunidad de la energía en Guatemala durante 2023 y 2024?

In [2]:
print(train["precio"].describe().round(2))
print("\nMediana por mes:")
print(train.groupby("mes")["precio"].median().round(1).rename(index=lambda m: meses_nom[m-1]))

count      17544.00
mean         284.21
std        12320.15
min            0.63
25%           76.43
50%          101.56
75%          132.10
max      1376899.00
Name: precio, dtype: float64

Mediana por mes:
mes
Ene     78.1
Feb     94.5
Mar    123.0
Abr    125.9
May    160.4
Jun    131.6
Jul    108.0
Ago     95.7
Sep     94.8
Oct     81.5
Nov     80.7
Dic     62.4
Name: precio, dtype: float64


Durante 2023 y 2024 el precio se mantuvo la mayor parte del tiempo alrededor de los 100 USD/MWh. La media es bastante más alta (284) porque la distribución tiene una cola larga hacia la derecha. Existen momentos en que el precio se dispara muy por encima de lo normal, con un máximo que supera el millón. Esos picos son pocos y se concentran sobre todo entre septiembre y octubre de 2023.

A lo largo del año el precio sigue un patrón estacional. Sube entre marzo y junio, siendo mayo el mes más caro, y baja en la segunda mitad del año, con diciembre como el más barato. Esto se relaciona con la dependencia de la generación hidroeléctrica: en época seca el precio es más alto y en época de lluvias baja.

Comparando ambos años, la forma estacional es casi la misma, pero el nivel general cambia un poco de un año al otro.

### 7.2 ¿Qué patrones horarios se observan en el mercado?

In [3]:
por_hora = train.groupby("hora")["precio"].median().round(1)
print(por_hora)
print("\nhora más barata:", por_hora.idxmin(), "| hora más cara:", por_hora.idxmax())

hora
0      79.5
1      77.6
2      76.9
3      76.5
4      77.7
5      81.8
6      89.1
7      97.9
8     104.9
9     108.7
10    110.6
11    113.8
12    115.3
13    115.1
14    115.3
15    117.3
16    117.9
17    117.9
18    124.6
19    126.3
20    122.4
21    115.0
22     91.1
23     81.3
Name: precio, dtype: float64

hora más barata: 3 | hora más cara: 19


Dentro de un mismo día el precio sigue un patrón regular. Como muestra la tabla, es más bajo de madrugada, con el mínimo alrededor de las 3, cuando el consumo es menor. A partir de ahí sube durante el día y llega a su punto más alto en la tarde/noche, con el máximo cerca de las 19. Después vuelve a bajar.

La dispersión también cambia según la hora: en las horas pico la banda entre el P25 y el P75 es más ancha, es decir que en esas horas el precio no solo es más alto sino también más variable.

Este patrón se repite todos los días de la semana, aunque se reduce el contraste el fin de semana.

### 7.3 ¿Existen horas pico, horas intermedias y horas valle? ¿Cómo las definieron?

In [4]:
mh = train.groupby("hora")["precio"].median()
c1, c2 = mh.quantile([1/3, 2/3])
print(f"umbrales (terciles de la mediana):  valle <= {c1:.1f}  <  intermedia <= {c2:.1f}  <  pico")

def clasificar(p):
    if p <= c1: return "valle"
    if p <= c2: return "intermedia"
    return "pico"

grupo = mh.apply(clasificar)
for g in ["valle", "intermedia", "pico"]:
    print(f"{g:11s} ->", sorted(grupo[grupo == g].index.tolist()))

umbrales (terciles de la mediana):  valle <= 90.5  <  intermedia <= 115.2  <  pico
valle       -> [0, 1, 2, 3, 4, 5, 6, 23]
intermedia  -> [7, 8, 9, 10, 11, 13, 21, 22]
pico        -> [12, 14, 15, 16, 17, 18, 19, 20]


Sí. Para definirlas se tomó el precio mediano de cada una de las 24 horas y se separaron en tres grupos según en qué tercio cae ese valor, por debajo de 90.5 USD/MWh son valle, entre 90.5 y 115.2 son intermedias, y por encima de 115.2 son pico.

Con ese criterio las horas valle son de 0 a 6 y la 23, las intermedias son de 7 a 11 más la 13, 21 y 22, y las pico son la 12 y de la 14 a la 20. Se usó la mediana y no la media para que los precios extremos de algunas horas no distorsionaran la clasificación.

### 7.4 ¿El precio parece más influenciado por la hora, el día de la semana o el mes?

In [ ]:
rangos = pd.DataFrame(index=["hora", "día de la semana", "mes"])
for nombre, col in zip(rangos.index, ["hora", "dia_sem", "mes"]):
    m = train.groupby(col)["precio"].median()
    rangos.loc[nombre, "min"] = m.min()
    rangos.loc[nombre, "max"] = m.max()
rangos["rango (max-min)"] = rangos["max"] - rangos["min"]
rangos.round(1)

,min,max,rango (max-min)
hora,76.5,126.3,49.9
día de la semana,82.6,109.2,26.6
mes,62.4,160.4,98.1


El mes es el que más influye. Mirando cuánto cambia el precio mediano dentro de cada factor, entre meses va de 62 a 160, entre horas va de 77 a 126, y entre días de la semana va de 83 a 109.

La variación por mes es casi el doble que la de la hora y casi cuatro veces la del día de la semana. Así que el factor estacional pesa más que el momento del día, y el día de la semana es el que menos influye.

### 7.5 ¿Qué tan estable es el comportamiento entre 2023 y 2024?

In [5]:
print(train.groupby("anio")["precio"].agg(["median", "mean", "max"]).round(1))

c23 = train[train["anio"] == 2023].groupby("mes")["precio"].median()
c24 = train[train["anio"] == 2024].groupby("mes")["precio"].median()
print("\ncorrelación de la forma mensual 2023 vs 2024:", round(c23.corr(c24), 2))

      median   mean        max
anio                          
2023   104.9  314.9  1376899.0
2024    96.4  253.6   810402.0

correlación de la forma mensual 2023 vs 2024: 0.6


El comportamiento es parecido pero no igual. Los dos años siguen la patrones similares estacionales, y la forma mensual de ambos está correlacionada de manera positiva. En contraste, la mediana de 2023 fue 104.9 y la de 2024 fue 96.4, por lo que el precio de 2023 estuvo más elevado. Los picos extremos también fueron mayores en 2023, los dos con outliers, pero 2023 con 1.4 millones contra 810 mil en 2024.
